# LeetCode 813 — Largest Sum of Averages bằng Python

Link đề: https://leetcode.com/problems/largest-sum-of-averages/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming (quy hoạch động), Prefix Sum (tổng tiền tố)

Skeleton LeetCode:

```python
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        ...
```

Ghi chú: kết quả là **số thực** và đề chấp nhận sai số `1e-6`. Vì vậy notebook này so sánh số thực **có dung sai** (`math.isclose`) và kiểm chứng bằng **`fractions.Fraction`** (số hữu tỉ chính xác, không sai số). Về thuật toán: đây là bài **chia mảng thành tối đa `k` đoạn liên tiếp** để tối đa tổng các trung bình đoạn; trạng thái DP là `(vị trí, số nhóm)` và tổng tiền tố cho trung bình đoạn trong `O(1)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia `nums` thành **tối đa `k`** đoạn liên tiếp không rỗng (dùng hết mọi phần tử), điểm số là **tổng trung bình** của các đoạn; tìm điểm lớn nhất.
2. Nắm các công cụ Python cần thiết: tổng tiền tố bằng `itertools.accumulate(..., initial=0)`, `functools.cache`/`lru_cache`, so sánh số thực bằng `math.isclose`, `fractions.Fraction`, `itertools.combinations` (chọn điểm cắt).
3. Hiểu hai ý chính: **công thức trạng thái** `f[i][j]` = điểm tốt nhất cho `i` phần tử đầu chia thành `j` đoạn, và **nhận xét** "chia nhiều đoạn hơn không bao giờ làm điểm giảm" (nên dùng đúng `k` đoạn).
4. Đi qua 5 cách: thử mọi tập điểm cắt, đệ quy có nhớ, DP bảng 2 chiều, DP một chiều (tiết kiệm bộ nhớ), DP bằng `Fraction` (chính xác); đối chiếu bằng oracle vét cạn dùng `Fraction`.
5. Tránh các lỗi kinh điển: dùng `//` thay `/`, lệch 1 với tổng tiền tố, tham lam cắt theo "lợi ích lớn nhất", quên điều kiện biên số đoạn, và so sánh số thực bằng `==`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Bạn có thể chia mảng thành **nhiều nhất `k`** đoạn con liền kề **không rỗng**. **Điểm** của một cách chia là **tổng các giá trị trung bình** của từng đoạn.

Lưu ý: cách chia phải dùng **mọi** phần tử của `nums`, và điểm không nhất thiết là số nguyên.

Trả về điểm **lớn nhất** trong mọi cách chia. Đáp án sai số không quá `10^-6` được chấp nhận.

**Ví dụ 1**

- Input: `nums = [9,1,2,3,9]`, `k = 3`
- Output: `20.00000`
- Giải thích: chia `[9]`, `[1, 2, 3]`, `[9]`: `9 + (1+2+3)/3 + 9 = 20`. Cách chia `[9, 1]`, `[2]`, `[3, 9]` cho `5 + 2 + 6 = 13`, kém hơn.

**Ví dụ 2**

- Input: `nums = [1,2,3,4,5,6,7]`, `k = 4`
- Output: `20.50000`

**Ràng buộc:**

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 10^4`
- `1 <= k <= nums.length`

In [1]:
from typing import List
from functools import cache
from itertools import accumulate, combinations
from fractions import Fraction
import math, random, time

examples = [
    ([9, 1, 2, 3, 9], 3, 20.0),
    ([1, 2, 3, 4, 5, 6, 7], 4, 20.5),
]
for nums, k, expected in examples:
    print(f"nums={nums} k={k} kỳ vọng={expected}")

nums=[9, 1, 2, 3, 9] k=3 kỳ vọng=20.0
nums=[1, 2, 3, 4, 5, 6, 7] k=4 kỳ vọng=20.5


## 3. Kiến thức nền

### 3.1. Tổng tiền tố (prefix sum)

`pre[i]` = tổng `i` phần tử đầu (`pre[0] = 0`). Khi đó tổng đoạn `nums[l..r-1]` (nửa mở `[l, r)`) là `pre[r] - pre[l]` và trung bình của nó là `(pre[r] - pre[l]) / (r - l)`. `accumulate(nums, initial=0)` tạo đúng `pre`.

In [2]:
nums = [9, 1, 2, 3, 9]
pre = list(accumulate(nums, initial=0))
print("pre =", pre)
l, r = 1, 4                                  # đoạn nums[1:4] = [1, 2, 3]
print(f"tổng nums[{l}:{r}] = {pre[r] - pre[l]}, trung bình = {(pre[r] - pre[l]) / (r - l)}")

pre = [0, 9, 10, 12, 15, 24]
tổng nums[1:4] = 6, trung bình = 2.0


### 3.2. Phép chia `/` và `//` trong Python 3

`/` luôn cho **số thực** (`7 / 2 = 3.5`), còn `//` là **chia lấy phần nguyên** (làm tròn xuống: `7 // 2 = 3`). Trung bình phải dùng `/`. Dùng nhầm `//` làm mất phần thập phân.

In [3]:
print(7 / 2, 7 // 2, 6 / 3, type(6 / 3))
print("trung bình [1,2,2]:", sum([1, 2, 2]) / 3, "(nhầm //:", sum([1, 2, 2]) // 3, ")")

3.5 3 2.0 <class 'float'>
trung bình [1,2,2]: 1.6666666666666667 (nhầm //: 1 )


### 3.3. Sai số của số thực và `math.isclose`

Số thực nhị phân không biểu diễn chính xác mọi số thập phân: `0.1 + 0.2 != 0.3`. Vì vậy **không so sánh số thực bằng `==`**. Dùng `math.isclose(a, b, abs_tol=1e-6)` (sai số tuyệt đối) hoặc `abs(a - b) < 1e-6`.

In [4]:
print(0.1 + 0.2, 0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3, abs_tol=1e-9))
print(math.isclose(20.000000001, 20.0, abs_tol=1e-6), math.isclose(20.01, 20.0, abs_tol=1e-6))

0.30000000000000004 False True
True False


### 3.4. `fractions.Fraction`: số hữu tỉ chính xác

`Fraction(a, b)` lưu `a/b` **chính xác** (tự rút gọn). Cộng, so sánh `Fraction` không có sai số. Ta dùng nó làm **oracle**: tính điểm đúng tuyệt đối, rồi so với kết quả `float` của thuật toán.

In [5]:
x = Fraction(1, 3) + Fraction(1, 6)
print(x, float(x), x == Fraction(1, 2))
print(Fraction(0.1) == Fraction(1, 10), "(Fraction(0.1) lấy đúng giá trị nhị phân của float, khác 1/10)")
total = sum((Fraction(s, d) for s, d in [(6, 3), (9, 1), (9, 1)]), Fraction(0))
print("9 + 2 + 9 =", total, float(total))

1/2 0.5 True
False (Fraction(0.1) lấy đúng giá trị nhị phân của float, khác 1/10)
9 + 2 + 9 = 20 20.0


### 3.5. Chọn điểm cắt bằng `combinations`

Chia `n` phần tử thành `m + 1` đoạn liên tiếp tương ứng chọn `m` điểm cắt trong `n - 1` khe (các khe `1..n-1`): `combinations(range(1, n), m)`. Số cách chia thành đúng `d` đoạn là `C(n - 1, d - 1)`.

In [6]:
n = 5
for m in range(0, 3):
    print(f"{m} điểm cắt:", list(combinations(range(1, n), m)))
print("tổng số cách chia mảng 5 phần tử thành tối đa 3 đoạn:", sum(math.comb(n - 1, m) for m in range(3)))

0 điểm cắt: [()]
1 điểm cắt: [(1,), (2,), (3,), (4,)]
2 điểm cắt: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
tổng số cách chia mảng 5 phần tử thành tối đa 3 đoạn: 11


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Với `nums = [9, 1, 2, 3, 9]`, `k = 3`: các giá trị 9 ở hai đầu **lớn** nên nên tách riêng để mỗi cái đóng góp cả `9` vào tổng; phần giữa `[1, 2, 3]` nhỏ, gộp lại thành trung bình 2. Điểm `9 + 2 + 9 = 20`.

Thử vài cách chia khác:

| cách chia | điểm |
|---|---|
| `[9,1,2,3,9]` (1 đoạn) | 4,8 |
| `[9] [1,2,3,9]` | 9 + 3,75 = 12,75 |
| `[9,1] [2] [3,9]` | 5 + 2 + 6 = 13 |
| `[9] [1,2,3] [9]` | 9 + 2 + 9 = **20** |

Nhận xét: **tách ra một đoạn thành hai** luôn không làm điểm giảm: trung bình của cả đoạn là bình quân gia quyền của hai nửa, nên **nhỏ hơn** tổng hai trung bình (vì số dương). Ví dụ `[2, 4]` có trung bình 3, tách thành `[2]` và `[4]` cho `2 + 4 = 6 > 3`. Vì vậy ta luôn nên dùng **đúng `k` đoạn** (khi `k <= n`).

In [7]:
def score(nums, bounds):
    """bounds: [0, c1, c2, ..., n]; trả điểm dưới dạng Fraction (chính xác)."""
    return sum(Fraction(sum(nums[bounds[i]:bounds[i + 1]]), bounds[i + 1] - bounds[i]) for i in range(len(bounds) - 1))

nums = [9, 1, 2, 3, 9]
for bounds in ([0, 5], [0, 1, 5], [0, 2, 3, 5], [0, 1, 4, 5]):
    parts = [nums[bounds[i]:bounds[i + 1]] for i in range(len(bounds) - 1)]
    print(f"{str(parts):28} điểm = {score(nums, bounds)} = {float(score(nums, bounds)):.5f}")

# tách một đoạn thành hai không làm điểm giảm
a, b = [2, 4], 3
print("trung bình [2,4] =", Fraction(6, 2), "< 2 + 4 =", 2 + 4)

[[9, 1, 2, 3, 9]]            điểm = 24/5 = 4.80000
[[9], [1, 2, 3, 9]]          điểm = 51/4 = 12.75000
[[9, 1], [2], [3, 9]]        điểm = 13 = 13.00000
[[9], [1, 2, 3], [9]]        điểm = 20 = 20.00000
trung bình [2,4] = 3 < 2 + 4 = 6


## 5. Cách 1 — Brute Force: thử mọi tập điểm cắt

**Ý tưởng.** Chọn `m` điểm cắt với `0 <= m <= k - 1` trong `n - 1` khe; mỗi tập cho một cách chia; tính điểm bằng tổng tiền tố; lấy max. Dùng `float` ở đây (oracle chính xác dùng `Fraction` ở mục sau).

In [8]:
def largest_brute(nums: List[int], k: int) -> float:
    n = len(nums)
    pre = list(accumulate(nums, initial=0))
    best = 0.0
    for m in range(0, k):                                  # m điểm cắt -> m + 1 đoạn
        for cuts in combinations(range(1, n), m):
            bounds = (0,) + cuts + (n,)
            total = sum((pre[bounds[i + 1]] - pre[bounds[i]]) / (bounds[i + 1] - bounds[i]) for i in range(m + 1))
            best = max(best, total)
    return best

for nums, k, expected in examples:
    got = largest_brute(nums, k)
    print(nums, k, "->", got, "OK" if math.isclose(got, expected, abs_tol=1e-6) else "SAI")
    assert math.isclose(got, expected, abs_tol=1e-6)

[9, 1, 2, 3, 9] 3 -> 20.0 OK
[1, 2, 3, 4, 5, 6, 7] 4 -> 20.5 OK


**Oracle chính xác (`Fraction`).** Cùng vét cạn nhưng mọi phép tính dùng `Fraction`, nên kết quả **không có sai số**. Dùng để so với mọi cách `float` khác.

In [9]:
def oracle(nums: List[int], k: int) -> Fraction:
    n = len(nums)
    pre = list(accumulate(nums, initial=0))
    best = Fraction(0)
    for m in range(0, k):
        for cuts in combinations(range(1, n), m):
            bounds = (0,) + cuts + (n,)
            total = sum((Fraction(pre[bounds[i + 1]] - pre[bounds[i]], bounds[i + 1] - bounds[i]) for i in range(m + 1)), Fraction(0))
            if total > best:
                best = total
    return best

def close(a, exact: Fraction, tol=1e-6) -> bool:
    return abs(Fraction(a) - exact) <= Fraction(tol)

for nums, k, expected in examples:
    ex = oracle(nums, k)
    print(nums, k, "-> chính xác", ex, "=", float(ex))
    assert close(expected, ex)

[9, 1, 2, 3, 9] 3 -> chính xác 20 = 20.0
[1, 2, 3, 4, 5, 6, 7] 4 -> chính xác 41/2 = 20.5


**Độ phức tạp và nút thắt.** Số cách chia là `sum_{m<k} C(n-1, m)`, tức tới `2^(n-1)` khi `k` lớn: `n = 100` là không thể. Hãy xem con số:

In [10]:
for n, k in [(10, 5), (20, 10), (40, 20), (100, 50), (100, 100)]:
    print(f"n={n:3} k={k:3}: số cách chia = {sum(math.comb(n - 1, m) for m in range(k)):,}")

n= 10 k=  5: số cách chia = 256
n= 20 k= 10: số cách chia = 262,144
n= 40 k= 20: số cách chia = 274,877,906,944
n=100 k= 50: số cách chia = 316,912,650,057,057,350,374,175,801,344
n=100 k=100: số cách chia = 633,825,300,114,114,700,748,351,602,688


Nút thắt: các cách chia chia sẻ **phần chung** (cùng đoạn đầu, khác đoạn sau), nên tính lại nhiều lần. Đây là dấu hiệu **bài toán con gối nhau** -> quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái.** `f[i][j]` = điểm lớn nhất khi chia `i` phần tử đầu (`nums[0..i-1]`) thành `j` đoạn. Đoạn cuối là `nums[h..i-1]` với `h` là điểm cắt cuối cùng (`j - 1 <= h <= i - 1`):

```
f[i][j] = max over h of ( f[h][j-1] + (pre[i] - pre[h]) / (i - h) )
f[i][1] = pre[i] / i          # một đoạn duy nhất
```

Đáp án: `f[n][k]`.

**Dùng đúng `k` đoạn.** Theo mục 4, chia thêm không làm điểm giảm; với `k <= n` luôn chia được đúng `k` đoạn nên đáp án "tối đa `k`" = đáp án "đúng `k`".

**Độ phức tạp.** Có `n * k` trạng thái, mỗi trạng thái thử `O(n)` điểm cắt, tổng `O(n^2 k)` = tối đa `100^3 = 10^6`: nhỏ.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy có nhớ `dfs(i, k)`: từ vị trí `i` còn `k` nhóm |
| 3 | DP bảng 2 chiều `f[i][j]` |
| 4 | DP một chiều: lặp theo số nhóm, chỉ giữ lớp trước |
| 5 | DP bằng `Fraction` để có kết quả **chính xác** rồi đổi sang `float` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ

**Giải thích từng đoạn.**

- `dfs(i, k)` = điểm lớn nhất khi chia **hậu tố** `nums[i:]` thành tối đa `k` đoạn.
- `i == n`: không còn phần tử -> `0`.
- `k == 1`: một đoạn duy nhất -> trung bình của hậu tố.
- Ngược lại thử mọi điểm kết thúc đoạn đầu: đoạn `nums[i:j]` với `j` từ `i + 1` tới `n - 1` (còn ít nhất một phần tử cho `k - 1` đoạn sau): `avg(i, j) + dfs(j, k - 1)`; lấy max.
- `@cache` nhớ kết quả theo `(i, k)`.

In [11]:
def largest_memo(nums: List[int], k: int) -> float:
    n = len(nums)
    pre = list(accumulate(nums, initial=0))

    @cache
    def dfs(i: int, k: int) -> float:
        if i == n:
            return 0
        if k == 1:
            return (pre[n] - pre[i]) / (n - i)
        ans = 0
        for j in range(i + 1, n):
            ans = max(ans, (pre[j] - pre[i]) / (j - i) + dfs(j, k - 1))
        return ans

    return dfs(0, k)

for nums, k, expected in examples:
    assert math.isclose(largest_memo(nums, k), expected, abs_tol=1e-6)
print("đệ quy có nhớ OK")

đệ quy có nhớ OK


**Code nộp LeetCode (Cách 2):**

In [12]:
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from functools import cache
        from itertools import accumulate

        @cache
        def dfs(i: int, k: int) -> float:
            if i == n:
                return 0
            if k == 1:
                return (s[n] - s[i]) / (n - i)
            ans = 0
            for j in range(i + 1, n):
                ans = max(ans, (s[j] - s[i]) / (j - i) + dfs(j, k - 1))
            return ans

        n = len(nums)
        s = list(accumulate(nums, initial=0))
        return dfs(0, k)

sol_memo = Solution().largestSumOfAverages
print([sol_memo(nums, k) for nums, k, _ in examples])

[20.0, 20.5]


**Độ phức tạp (Cách 2):** thời gian `O(n^2 k)`, bộ nhớ `O(n k)` (bộ nhớ đệm) cộng `O(k)` ngăn xếp đệ quy.

---

### 7.2. Cách 3 — DP bảng 2 chiều

**Giải thích từng đoạn.**

- `s` = tổng tiền tố.
- `f[i][1] = s[i] / i` với `i = 1..n`.
- Với `j` từ `2` tới `min(i, k)`: `f[i][j] = max(f[h][j-1] + (s[i] - s[h]) / (i - h))` với `h` từ `j - 1` tới `i - 1` (cần ít nhất `j - 1` phần tử cho `j - 1` đoạn đầu).
- Đáp án `f[n][k]`.

In [13]:
def largest_table(nums: List[int], k: int) -> float:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    f = [[0.0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        f[i][1] = s[i] / i
        for j in range(2, min(i, k) + 1):
            for h in range(j - 1, i):
                f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
    return f[n][k]

for nums, k, expected in examples:
    assert math.isclose(largest_table(nums, k), expected, abs_tol=1e-6)
print("DP bảng OK")

DP bảng OK


**Code nộp LeetCode (Cách 3):**

In [14]:
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from itertools import accumulate
        n = len(nums)
        s = list(accumulate(nums, initial=0))
        f = [[0.0] * (k + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            f[i][1] = s[i] / i
            for j in range(2, min(i, k) + 1):
                for h in range(j - 1, i):
                    f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
        return f[n][k]

sol_table = Solution().largestSumOfAverages
print([sol_table(nums, k) for nums, k, _ in examples])

[20.0, 20.5]


**Độ phức tạp (Cách 3):** thời gian `O(n^2 k)`, bộ nhớ `O(n k)`.

---

### 7.3. Cách 4 — DP một chiều theo số nhóm

**Ý tưởng khác.** `f[i][j]` chỉ phụ thuộc lớp `j - 1`. Vậy giữ **một mảng `dp` độ dài `n + 1`** và cập nhật `k - 1` lần, mỗi lần tính lớp mới từ lớp cũ.

**Giải thích từng đoạn.**

- Lớp 1: `dp[i] = s[i] / i` (`dp[0]` không dùng).
- Mỗi lần lặp thêm một nhóm (nhóm thứ `j = 2..k`): `new[i] = max over h in [j-1, i-1] of dp[h] + (s[i] - s[h])/(i - h)` cho `i >= j`; các `i < j` đặt `-inf` (không chia được đủ `j` đoạn).
- Sau `k - 1` lần, `dp[n]` là đáp án (đúng `k` đoạn, bằng "tối đa `k`").

In [15]:
def largest_rolling(nums: List[int], k: int) -> float:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    NEG = float("-inf")
    dp = [NEG] + [s[i] / i for i in range(1, n + 1)]          # đúng 1 đoạn
    for j in range(2, k + 1):
        new = [NEG] * (n + 1)
        for i in range(j, n + 1):
            best = NEG
            for h in range(j - 1, i):
                cand = dp[h] + (s[i] - s[h]) / (i - h)
                if cand > best:
                    best = cand
            new[i] = best
        dp = new
    return dp[n]

for nums, k, expected in examples:
    assert math.isclose(largest_rolling(nums, k), expected, abs_tol=1e-6)
print("DP một chiều OK")

DP một chiều OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from itertools import accumulate
        n = len(nums)
        s = list(accumulate(nums, initial=0))
        NEG = float("-inf")
        dp = [NEG] + [s[i] / i for i in range(1, n + 1)]
        for j in range(2, k + 1):
            new = [NEG] * (n + 1)
            for i in range(j, n + 1):
                for h in range(j - 1, i):
                    cand = dp[h] + (s[i] - s[h]) / (i - h)
                    if cand > new[i]:
                        new[i] = cand
            dp = new
        return dp[n]

sol_rolling = Solution().largestSumOfAverages
print([sol_rolling(nums, k) for nums, k, _ in examples])

[20.0, 20.5]


**Độ phức tạp (Cách 4):** thời gian `O(n^2 k)`, bộ nhớ `O(n)`.

---

### 7.4. Cách 5 — DP bằng `Fraction` (chính xác)

**Ý tưởng khác.** Cùng công thức nhưng mọi giá trị là `Fraction`, nên **không có sai số tích lũy**. Kết quả `Fraction` đúng tuyệt đối rồi đổi sang `float` một lần ở cuối. Chậm hơn `float` vài chục lần nhưng với `n <= 100` vẫn chạy được, và là **chuẩn vàng** để kiểm các cách `float`.

**Giải thích từng đoạn.** Giống Cách 4; thay `/` bằng `Fraction(a, b)`.

In [17]:
def largest_exact(nums: List[int], k: int) -> Fraction:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    dp = [None] + [Fraction(s[i], i) for i in range(1, n + 1)]
    for j in range(2, k + 1):
        new = [None] * (n + 1)
        for i in range(j, n + 1):
            best = None
            for h in range(j - 1, i):
                cand = dp[h] + Fraction(s[i] - s[h], i - h)
                if best is None or cand > best:
                    best = cand
            new[i] = best
        dp = new
    return dp[n]

for nums, k, expected in examples:
    ex = largest_exact(nums, k)
    print(nums, k, "->", ex, "=", float(ex))
    assert ex == oracle(nums, k) and close(expected, ex)
print("DP Fraction OK")

[9, 1, 2, 3, 9] 3 -> 20 = 20.0
[1, 2, 3, 4, 5, 6, 7] 4 -> 41/2 = 20.5
DP Fraction OK


**Code nộp LeetCode (Cách 5):** trả `float` (LeetCode so sai số 1e-6); chậm hơn nhưng vẫn nằm trong giới hạn với `n <= 100`.

In [18]:
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from fractions import Fraction
        from itertools import accumulate
        n = len(nums)
        s = list(accumulate(nums, initial=0))
        dp = [None] + [Fraction(s[i], i) for i in range(1, n + 1)]
        for j in range(2, k + 1):
            new = [None] * (n + 1)
            for i in range(j, n + 1):
                best = None
                for h in range(j - 1, i):
                    cand = dp[h] + Fraction(s[i] - s[h], i - h)
                    if best is None or cand > best:
                        best = cand
                new[i] = best
            dp = new
        return float(dp[n])

sol_exact = Solution().largestSumOfAverages
print([sol_exact(nums, k) for nums, k, _ in examples])

[20.0, 20.5]


**Độ phức tạp (Cách 5):** thời gian `O(n^2 k)` phép `Fraction` (mỗi phép chậm hơn `float`), bộ nhớ `O(n)` số hữu tỉ (mẫu số có thể lớn nhưng chỉ là bội chung của các độ dài đoạn `<= 100`).

## 8. TRACE từng bước

### 8.1. Bảng `f[i][j]` cho `nums = [9, 1, 2, 3, 9]`, `k = 3`

Tổng tiền tố `s = [0, 9, 10, 12, 15, 24]`.

| `i` \ `j` | 1 | 2 | 3 |
|---|---|---|---|
| 1 | 9 | | |
| 2 | 5 | 10 | |
| 3 | 4 | 10,5 | 12 |
| 4 | 3,75 | 11 | 13,5 |
| 5 | 4,8 | 12,75 | **20** |

Ví dụ `f[5][3]`: thử `h = 2, 3, 4`: `f[2][2] + avg(nums[2..4]) = 10 + 14/3 ≈ 14,67`; `f[3][2] + avg(nums[3..4]) = 10,5 + 6 = 16,5`; `f[4][2] + nums[4] = 11 + 9 = 20` (đây là cách chia `[9] [1,2,3] [9]`). Lấy max -> `20`. Hàm `trace_table` in bảng và từng ứng viên.

In [19]:
def trace_table(nums, k, detail_cell=None):
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    print("s =", s)
    f = [[0.0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        f[i][1] = s[i] / i
        for j in range(2, min(i, k) + 1):
            cands = []
            for h in range(j - 1, i):
                val = f[h][j - 1] + (s[i] - s[h]) / (i - h)
                cands.append((h, val))
                f[i][j] = max(f[i][j], val)
            if detail_cell == (i, j):
                print(f"  chi tiết f[{i}][{j}]:", [(h, round(v, 4)) for h, v in cands], "-> max", round(f[i][j], 4))
    print("bảng f (hàng i = số phần tử, cột j = số đoạn):")
    for i in range(1, n + 1):
        print(f"  i={i}: " + "  ".join(f"{f[i][j]:8.4f}" if (j <= i) else "        " for j in range(1, k + 1)))
    return f[n][k]

ans = trace_table([9, 1, 2, 3, 9], 3, detail_cell=(5, 3))
assert math.isclose(ans, 20.0, abs_tol=1e-9)

s = [0, 9, 10, 12, 15, 24]
  chi tiết f[5][3]: [(2, 14.6667), (3, 16.5), (4, 20.0)] -> max 20.0
bảng f (hàng i = số phần tử, cột j = số đoạn):
  i=1:   9.0000                    
  i=2:   5.0000   10.0000          
  i=3:   4.0000   10.5000   12.0000
  i=4:   3.7500   11.0000   13.5000
  i=5:   4.8000   12.7500   20.0000


Ta cũng có thể kiểm tra **từng ô** của bảng bằng oracle chính xác: `f[i][j]` phải bằng điểm tốt nhất khi chia **đúng** `j` đoạn cho `i` phần tử đầu (ở đây các ô đều cho kết quả "đúng `j` đoạn" vì chia thêm không làm giảm điểm).

In [20]:
# kiểm tra tay các ô của bảng bằng oracle chính xác (f[i][j] = oracle với prefix i và đúng j đoạn)
def exact_for_prefix(nums, i, j):
    return max((score(nums[:i], (0,) + cuts + (i,)) for cuts in combinations(range(1, i), j - 1)), default=None)

nums = [9, 1, 2, 3, 9]
for i in range(1, 6):
    row = []
    for j in range(1, min(i, 3) + 1):
        row.append(float(exact_for_prefix(nums, i, j)))
    print(f"i={i}:", [round(v, 4) for v in row])

i=1: [9.0]
i=2: [5.0, 10.0]
i=3: [4.0, 10.5, 12.0]
i=4: [3.75, 11.0, 13.5]
i=5: [4.8, 12.75, 20.0]


### 8.2. Truy vết lựa chọn điểm cắt

Lưu thêm `choice[i][j] = h` (điểm cắt tốt nhất) rồi đi ngược từ `(n, k)` để biết cách chia tối ưu. Với ví dụ 1 ta kỳ vọng `[9] | [1,2,3] | [9]`.

In [21]:
def best_partition(nums, k):
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    NEG = float("-inf")
    f = [[NEG] * (k + 1) for _ in range(n + 1)]
    choice = [[-1] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        f[i][1] = s[i] / i
    for j in range(2, k + 1):
        for i in range(j, n + 1):
            for h in range(j - 1, i):
                val = f[h][j - 1] + (s[i] - s[h]) / (i - h)
                if val > f[i][j]:
                    f[i][j], choice[i][j] = val, h
    bounds, i, j = [n], n, k
    while j > 1:
        i = choice[i][j]
        bounds.append(i)
        j -= 1
    bounds.append(0)
    bounds.reverse()
    return f[n][k], bounds

value, bounds = best_partition([9, 1, 2, 3, 9], 3)
parts = [[9, 1, 2, 3, 9][bounds[i]:bounds[i + 1]] for i in range(len(bounds) - 1)]
print("điểm =", value, "| cách chia:", parts)
assert math.isclose(value, 20.0, abs_tol=1e-9) and parts == [[9], [1, 2, 3], [9]]

điểm = 20.0 | cách chia: [[9], [1, 2, 3], [9]]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (đệ quy có nhớ) hoặc **Cách 3** (bảng): cả hai `O(n^2 k)`; bản dưới nộp **Cách 3** (không đệ quy, không phụ thuộc giới hạn ngăn xếp).

```python
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from itertools import accumulate
        n = len(nums)
        s = list(accumulate(nums, initial=0))
        f = [[0.0] * (k + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            f[i][1] = s[i] / i
            for j in range(2, min(i, k) + 1):
                for h in range(j - 1, i):
                    f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
        return f[n][k]
```

In [22]:
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from itertools import accumulate
        n = len(nums)
        s = list(accumulate(nums, initial=0))
        f = [[0.0] * (k + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            f[i][1] = s[i] / i
            for j in range(2, min(i, k) + 1):
                for h in range(j - 1, i):
                    f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
        return f[n][k]

sol = Solution()
for nums, k, expected in examples:
    got = sol.largestSumOfAverages(nums, k)
    print(nums, k, "->", got, "OK" if math.isclose(got, expected, abs_tol=1e-6) else "SAI")
    assert math.isclose(got, expected, abs_tol=1e-6)

[9, 1, 2, 3, 9] 3 -> 20.0 OK
[1, 2, 3, 4, 5, 6, 7] 4 -> 20.5 OK


## 10. Vì sao đúng

**(a) Cấu trúc con tối ưu.** Xét một cách chia tối ưu của `nums[0..i-1]` thành `j` đoạn, đoạn cuối là `nums[h..i-1]`. Phần còn lại `nums[0..h-1]` phải là cách chia tối ưu thành `j - 1` đoạn: nếu có cách chia tốt hơn thì thay vào được cách chia cả mảng tốt hơn, mâu thuẫn. Vậy `f[i][j] = max_h ( f[h][j-1] + avg(h, i) )`: duyệt mọi `h` hợp lệ.

**(b) Tối đa `k` đoạn = đúng `k` đoạn.** Với hai số dương `a, b` (số phần tử `p, q`): trung bình đoạn gộp `(p*A + q*B)/(p + q)` với `A, B` là trung bình hai nửa, và `(p*A + q*B)/(p+q) <= max(A, B) < A + B` (vì `A, B > 0`). Nên tách một đoạn thành hai đoạn **luôn làm điểm tăng thực sự**. Do đó, nếu cách chia dùng `< k` đoạn và còn có thể tách (đoạn dài >= 2 tồn tại khi `số đoạn < n`), ta tách thêm để điểm tăng; lặp lại tới khi đủ `k` đoạn (được vì `k <= n`). Vậy tối ưu luôn đạt ở đúng `k` đoạn; các cách DP "đúng `k`" (Cách 4, 5) và "tối đa `k`" (Cách 2, 3) cho cùng kết quả.

**(c) Sai số.** Cách float cộng tối đa `k <= 100` số có độ lớn `<= 10^4` (kết quả `<= 10^6`): sai số tích lũy cỡ `10^-10`, nhỏ hơn nhiều so với yêu cầu `10^-6`. Ta kiểm chứng bằng `Fraction` rằng `|float - chính xác| < 10^-9` trên nhiều input.

**Kiểm chứng bằng máy:** so các cách `float` với oracle `Fraction` vét cạn trên **mọi** mảng nhỏ (độ dài tới 7, giá trị 1..4, mọi `k`) và trên nhiều mảng ngẫu nhiên; với `n` lớn (tới 100) so các cách `float` với DP `Fraction`.

In [23]:
solutions = {
    "brute": largest_brute,
    "memo": largest_memo,
    "table": largest_table,
    "rolling": largest_rolling,
    "exact": lambda nums, k: float(largest_exact(nums, k)),
    "sol_memo": sol_memo,
    "sol_table": sol_table,
    "sol_rolling": sol_rolling,
    "sol_exact": sol_exact,
    "final": Solution().largestSumOfAverages,
}

checked, worst = 0, 0.0
for n in range(1, 8):
    for arr in __import__("itertools").product(range(1, 5), repeat=n):
        nums = list(arr)
        for k in range(1, n + 1):
            ref = oracle(nums, k)
            for name, fn in solutions.items():
                got = fn(nums, k)
                err = abs(Fraction(got) - ref)
                worst = max(worst, float(err))
                assert err < Fraction(1, 10 ** 9), (name, nums, k, got, ref)
            checked += 1
print(f"mọi mảng độ dài 1..7 (giá trị 1..4), mọi k: {checked} ca x {len(solutions)} cài đặt khớp oracle Fraction; sai số lớn nhất {worst:.2e}")

mọi mảng độ dài 1..7 (giá trị 1..4), mọi k: 145636 ca x 10 cài đặt khớp oracle Fraction; sai số lớn nhất 2.37e-15


## 11. Time / Space Complexity

Gọi `n = len(nums)` (`<= 100`) và `k <= n`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập điểm cắt | `O(2^n * k)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n^2 k)` | `O(n k)` |
| 3. DP bảng | `O(n^2 k)` | `O(n k)` |
| 4. DP một chiều | `O(n^2 k)` | `O(n)` |
| 5. DP `Fraction` | `O(n^2 k)` phép số hữu tỉ | `O(n)` |

Trường hợp xấu nhất `n = k = 100` là cỡ `10^6 / 6` phép (đã trừ các ô vô nghĩa) — vài chục tới vài trăm mili-giây trong Python; `Fraction` chậm hơn hẳn. Đo thử:

In [24]:
rng = random.Random(813)
nums_big = [rng.randint(1, 10 ** 4) for _ in range(100)]
exact_ref = None
for name in ["table", "rolling", "memo", "exact"]:
    fn = solutions[name]
    t0 = time.perf_counter()
    val = fn(nums_big, 100)
    dt = (time.perf_counter() - t0) * 1000
    if exact_ref is None:
        exact_ref = val
    assert math.isclose(val, exact_ref, abs_tol=1e-6)
    print(f"{name:8} n=100, k=100: {val:.6f}  {dt:9.1f} ms")
t0 = time.perf_counter()
v50 = solutions["table"](nums_big, 50)
print(f"table    n=100, k=50 : {v50:.6f}  {1000 * (time.perf_counter() - t0):9.1f} ms")
print("k = n: điểm bằng tổng của chính các phần tử:", sum(nums_big), "-> khớp:", math.isclose(exact_ref, sum(nums_big), abs_tol=1e-6))

table    n=100, k=100: 493861.000000       21.9 ms
rolling  n=100, k=100: 493861.000000       12.4 ms
memo     n=100, k=100: 493861.000000       31.0 ms


exact    n=100, k=100: 493861.000000      249.1 ms
table    n=100, k=50 : 328576.266667       16.4 ms
k = n: điểm bằng tổng của chính các phần tử: 493861 -> khớp: True


## 12. Bộ test

Gồm: ví dụ của đề (so sánh có dung sai), edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle `Fraction`, và test ngẫu nhiên có seed cố định (đúng ràng buộc).

In [25]:
# 12.1 ví dụ đề (so sánh số thực có dung sai 1e-6)
for nums, k, expected in examples:
    for name, fn in solutions.items():
        assert math.isclose(fn(nums, k), expected, abs_tol=1e-6), (name, nums, k)
print("12.1 ví dụ đề: OK")

12.1 ví dụ đề: OK


In [26]:
edge = [
    ([5], 1, 5.0),                                   # một phần tử
    ([1, 2, 3], 1, 2.0),                             # k = 1: trung bình cả mảng
    ([1, 2, 3], 3, 6.0),                             # k = n: tổng các phần tử
    ([4, 4, 4, 4], 2, 8.0),                          # mọi phần tử bằng nhau: k * giá trị
    ([4, 4, 4, 4], 4, 16.0),
    ([1, 100], 2, 101.0),
    ([1, 100], 1, 50.5),
    ([10000] * 100, 100, 1000000.0),                 # giá trị và độ dài lớn nhất
    ([10000] * 100, 1, 10000.0),
    ([1] * 100, 7, 7.0),
    (list(range(1, 8)), 4, 20.5),                    # ví dụ 2
    ([1, 2], 2, 3.0),
    ([3, 1, 2], 2, 4.5),                             # [3] [1,2]
    ([1, 2, 3, 4], 2, 6.0),                          # [1,2,3] [4] = 2 + 4
]
for nums, k, expected in edge:
    if len(nums) <= 12:
        assert close(expected, oracle(nums, k), 1e-12), ("oracle khác kỳ vọng", nums, k)
    for name, fn in solutions.items():
        if name == "brute" and len(nums) > 12:
            continue
        assert math.isclose(fn(nums, k), expected, abs_tol=1e-6), (name, nums, k, fn(nums, k), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.2 edge case: OK 14 ca


In [27]:
# 12.3 phản ví dụ: các cách sai
def wrong_floor_div(nums, k):          # dùng // thay cho /
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    f = [[0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        f[i][1] = s[i] // i
        for j in range(2, min(i, k) + 1):
            for h in range(j - 1, i):
                f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) // (i - h))
    return f[n][k]

def wrong_prefix_off_by_one(nums, k):  # s[i] - s[h] nhưng chia cho (i - h + 1)
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    f = [[0.0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        f[i][1] = s[i] / i
        for j in range(2, min(i, k) + 1):
            for h in range(j - 1, i):
                f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h + 1))
    return f[n][k]

def wrong_greedy_gain(nums, k):        # mỗi lần chia chỗ có lợi nhất ngay lúc đó (k - 1 lần)
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    avg = lambda a, b: Fraction(s[b] - s[a], b - a)
    bounds = [0, n]
    for _ in range(k - 1):
        best = None
        for t in range(len(bounds) - 1):
            a, b = bounds[t], bounds[t + 1]
            for c in range(a + 1, b):
                gain = avg(a, c) + avg(c, b) - avg(a, b)
                if best is None or gain > best[0]:
                    best = (gain, c)
        if best is None:
            break
        bounds = sorted(bounds + [best[1]])
    return float(sum(avg(bounds[t], bounds[t + 1]) for t in range(len(bounds) - 1)))

def wrong_no_base(nums, k):            # quên f[i][1]: cột j = 1 giữ ở 0
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    f = [[0.0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(2, min(i, k) + 1):
            for h in range(j - 1, i):
                f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
    return f[n][k]

wrongs = [
    ("dùng // (chia nguyên)", wrong_floor_div, [1, 2, 2], 1),
    ("lệch 1 ở mẫu số", wrong_prefix_off_by_one, [9, 1, 2, 3, 9], 3),
    ("tham lam chia lợi nhất", wrong_greedy_gain, [6, 2, 8, 9, 2, 3, 9], 4),
    ("quên trường hợp 1 đoạn", wrong_no_base, [1, 2, 3], 2),
]
for name, fn, nums, k in wrongs:
    truth = oracle(nums, k)
    got = fn(nums, k)
    print(f"{name:26} nums={nums} k={k}: sai = {float(got):.4f}, đúng = {float(truth):.4f}")
    assert not close(got, truth, 1e-6), name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 so sánh == với số thực rất dễ sai: cùng một đáp án nhưng tính theo hai thứ tự cộng khác nhau
a = 0.1 + 0.2 + 0.3
b = 0.3 + 0.2 + 0.1
print("0.1+0.2+0.3 == 0.3+0.2+0.1 ?", a == b, "| isclose:", math.isclose(a, b, abs_tol=1e-9))

dùng // (chia nguyên)      nums=[1, 2, 2] k=1: sai = 1.0000, đúng = 1.6667
lệch 1 ở mẫu số            nums=[9, 1, 2, 3, 9] k=3: sai = 15.0000, đúng = 20.0000
tham lam chia lợi nhất     nums=[6, 2, 8, 9, 2, 3, 9] k=4: sai = 24.6667, đúng = 25.8333
quên trường hợp 1 đoạn     nums=[1, 2, 3] k=2: sai = 3.0000, đúng = 4.5000
12.3 phản ví dụ: OK (các cách sai đều bị lộ)
0.1+0.2+0.3 == 0.3+0.2+0.1 ? False | isclose: True


In [28]:
# 12.5 ngẫu nhiên nhỏ so với oracle Fraction (n <= 10)
rng = random.Random(2813)
worst = 0.0
for _ in range(400):
    n = rng.randint(1, 10)
    nums = [rng.randint(1, rng.choice([5, 100, 10 ** 4])) for _ in range(n)]
    k = rng.randint(1, n)
    ref = oracle(nums, k)
    for name, fn in solutions.items():
        got = fn(nums, k)
        err = abs(Fraction(got) - ref)
        worst = max(worst, float(err))
        assert err < Fraction(1, 10 ** 6), (name, nums, k, got, float(ref))
print(f"12.5 ngẫu nhiên n <= 10: OK, sai số lớn nhất {worst:.2e}")

# 12.6 ngẫu nhiên lớn (n tới 100): các cách float khớp DP Fraction (chính xác) trong 1e-6
rng = random.Random(77)
worst = 0.0
for _ in range(25):
    n = rng.randint(30, 100)
    k = rng.randint(1, n)
    nums = [rng.randint(1, 10 ** 4) for _ in range(n)]
    exact = largest_exact(nums, k)
    for name in ("table", "rolling", "memo", "sol_table", "sol_memo", "final"):
        got = solutions[name](nums, k)
        err = abs(Fraction(got) - exact)
        worst = max(worst, float(err))
        assert err < Fraction(1, 10 ** 6), (name, n, k, got, float(exact))
print(f"12.6 ngẫu nhiên n tới 100 so với DP Fraction: OK, sai số lớn nhất {worst:.2e}")

# 12.7 tính chất: điểm không giảm khi tăng k; k = n cho tổng; k = 1 cho trung bình
rng = random.Random(9)
for _ in range(60):
    n = rng.randint(1, 20)
    nums = [rng.randint(1, 100) for _ in range(n)]
    vals = [largest_exact(nums, k) for k in range(1, n + 1)]
    assert all(vals[i] <= vals[i + 1] for i in range(n - 1)), nums
    assert vals[0] == Fraction(sum(nums), n) and vals[-1] == sum(nums)
print("12.7 tính chất đơn điệu theo k, k = 1, k = n: OK")

12.5 ngẫu nhiên n <= 10: OK, sai số lớn nhất 2.43e-12


12.6 ngẫu nhiên n tới 100 so với DP Fraction: OK, sai số lớn nhất 6.10e-11


12.7 tính chất đơn điệu theo k, k = 1, k = n: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `//` thay cho `/`

Trung bình cần số thực; `//` cắt phần thập phân.

In [29]:
nums, k = [1, 2, 2], 1
print("sai (//):", wrong_floor_div(nums, k), "| đúng:", Solution().largestSumOfAverages(nums, k))
assert wrong_floor_div(nums, k) == 1 and math.isclose(Solution().largestSumOfAverages(nums, k), 5 / 3, abs_tol=1e-9)

sai (//): 1 | đúng: 1.6666666666666667


Vì sao sai: `5 // 3 = 1` còn trung bình thật là `1,6667...`. **Đúng:** `/`.

### Lỗi 2 — Lệch 1 ở độ dài đoạn

Đoạn nửa mở `[h, i)` có `i - h` phần tử; chia cho `i - h + 1` là sai.

In [30]:
nums, k = [9, 1, 2, 3, 9], 3
print("sai (i - h + 1):", round(wrong_prefix_off_by_one(nums, k), 4), "| đúng:", Solution().largestSumOfAverages(nums, k))
assert not math.isclose(wrong_prefix_off_by_one(nums, k), 20.0, abs_tol=1e-6)

sai (i - h + 1): 15.0 | đúng: 20.0


Vì sao sai: với tổng tiền tố `s`, tổng đoạn `[h, i)` là `s[i] - s[h]` và số phần tử là `i - h`. **Đúng:** chia cho `i - h`.

### Lỗi 3 — Tham lam: chia ở chỗ lợi nhất ngay lúc đó

Cố "mỗi lần chia chỗ tăng điểm nhiều nhất" nghe hợp lý nhưng **không tối ưu toàn cục**.

In [31]:
nums, k = [6, 2, 8, 9, 2, 3, 9], 4
print("tham lam:", round(wrong_greedy_gain(nums, k), 4), "| đúng:", round(float(oracle(nums, k)), 4))
assert wrong_greedy_gain(nums, k) < float(oracle(nums, k)) - 1e-9

tham lam: 24.6667 | đúng: 25.8333


Vì sao sai: một lần chia tối ưu về sau có thể cần điểm cắt mà lúc đầu trông kém. Bài toán có cấu trúc con tối ưu nhưng **không** có tính chất tham lam. **Đúng:** DP.

### Lỗi 4 — Quên trường hợp một đoạn (`f[i][1]`)

Nếu không đặt `f[i][1] = s[i] / i` thì cột `j = 1` giữ ở 0, nên mỗi ô chỉ còn **trung bình của đoạn cuối** (mất phần của các đoạn đầu).

In [32]:
nums, k = [1, 2, 3], 2
print("sai (thiếu f[i][1]):", wrong_no_base(nums, k), "| đúng:", Solution().largestSumOfAverages(nums, k))
assert wrong_no_base(nums, k) == 3.0 and math.isclose(Solution().largestSumOfAverages(nums, k), 4.5, abs_tol=1e-9)

sai (thiếu f[i][1]): 3.0 | đúng: 4.5


Vì sao sai: `f[h][j-1]` ở `j = 2` chính là `f[h][1]`: nếu nó bằng 0 thì chỉ còn trung bình đoạn cuối. **Đúng:** khởi tạo cột `j = 1`.

### Lỗi 5 — So sánh số thực bằng `==`

Hai cách tính cùng đáp án có thể lệch ở chữ số cuối do thứ tự cộng khác nhau.

In [33]:
x = 0.0
for _ in range(10):
    x += 0.1                    # cộng lần lượt (sum() của Python 3.12+ bù sai số nên che mất hiện tượng)
print(x, x == 1.0, math.isclose(x, 1.0, abs_tol=1e-9))
r1 = largest_table([1, 2, 3, 4, 5, 6, 7], 4)
r2 = largest_rolling([1, 2, 3, 4, 5, 6, 7], 4)
print(r1, r2, "bằng nhau?", r1 == r2, "| isclose:", math.isclose(r1, r2, abs_tol=1e-9))
assert x != 1.0

0.9999999999999999 False True
20.5 20.5 bằng nhau? True | isclose: True


Vì sao sai: số thực có sai số làm tròn; LeetCode cũng chỉ yêu cầu `1e-6`. **Đúng:** `math.isclose(a, b, abs_tol=1e-6)` hoặc `abs(a - b) < 1e-6`; muốn chính xác tuyệt đối dùng `Fraction`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia mảng thành tối đa `k` đoạn liền kề để tối ưu một hàm cộng theo từng đoạn" -> `f[i][j]` = tốt nhất cho `i` phần tử đầu, `j` đoạn; chuyển trạng thái bằng điểm cắt cuối `h`.
- **Tổng tiền tố** biến mọi truy vấn "tổng/trung bình đoạn" thành `O(1)`.
- **Nhận xét đơn điệu:** khi hàm điểm chỉ tăng khi chia nhỏ hơn thì "tối đa `k`" = "đúng `k`" (giảm số trạng thái hợp lệ).
- Cùng họ: Split Array Largest Sum (410), Palindrome Partitioning III, Minimum Cost to Cut a Stick, Partition Array for Maximum Sum (1043).
- Với kết quả số thực: giữ **toàn số nguyên/`Fraction`** khi có thể, chỉ chuyển `float` ở bước cuối; so sánh bằng dung sai.
- Cẩn thận biên: `j - 1 <= h <= i - 1` (đủ phần tử cho `j - 1` đoạn đầu).
- Test nên có: `k = 1`, `k = n`, mọi phần tử bằng nhau, hai giá trị chênh lệch rất lớn, `n = 100` giá trị `10^4`.

## 15. Biến thể

1. **Trả về cách chia** (các điểm cắt) cùng điểm: đã làm ở mục 8.2 (truy vết `choice`).
2. **Tổng trung vị** thay vì trung bình: DP cùng khung nhưng chi phí đoạn tính khác (cần cấu trúc dữ liệu cho trung vị).
3. **Tối thiểu hóa** tổng trung bình với đúng `k` đoạn: đổi `max` thành `min` (bây giờ "đúng `k`" mới có nghĩa).
4. Bài liên quan: Split Array Largest Sum (410, DP hoặc chặt nhị phân), Partition Array for Maximum Sum (1043).

Dưới đây là code cho biến thể 3 (đúng `k` đoạn, tối thiểu) và kiểm bằng oracle `Fraction`.

In [34]:
def smallest_sum_of_averages_exact_k(nums: List[int], k: int) -> Fraction:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    dp = [None] + [Fraction(s[i], i) for i in range(1, n + 1)]
    for j in range(2, k + 1):
        new = [None] * (n + 1)
        for i in range(j, n + 1):
            new[i] = min(dp[h] + Fraction(s[i] - s[h], i - h) for h in range(j - 1, i))
        dp = new
    return dp[n]

def smallest_oracle(nums, k):
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    return min(sum((Fraction(s[b[i + 1]] - s[b[i]], b[i + 1] - b[i]) for i in range(k)), Fraction(0))
               for cuts in combinations(range(1, n), k - 1) for b in [(0,) + cuts + (n,)])

rng = random.Random(15)
for _ in range(150):
    n = rng.randint(1, 9)
    nums = [rng.randint(1, 30) for _ in range(n)]
    k = rng.randint(1, n)
    assert smallest_sum_of_averages_exact_k(nums, k) == smallest_oracle(nums, k), (nums, k)
print("ví dụ:", smallest_sum_of_averages_exact_k([9, 1, 2, 3, 9], 3), "(nhỏ nhất với đúng 3 đoạn)")
print("Biến thể: OK")

ví dụ: 13 (nhỏ nhất với đúng 3 đoạn)
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập cắt | Định nghĩa trực tiếp | `O(2^n k)` | `O(n)` | Oracle, `n` nhỏ |
| 2. Đệ quy có nhớ | `dfs(i, k)` trên hậu tố | `O(n^2 k)` | `O(n k)` | Dễ suy ra từ đệ quy |
| 3. DP bảng | `f[i][j]` trên tiền tố | `O(n^2 k)` | `O(n k)` | **Cách nộp chính**, không đệ quy |
| 4. DP một chiều | Lặp theo số nhóm, giữ lớp trước | `O(n^2 k)` | `O(n)` | Tiết kiệm bộ nhớ |
| 5. DP `Fraction` | Số hữu tỉ chính xác | `O(n^2 k)` (chậm hằng số) | `O(n)` | Kiểm chứng; khi cần độ chính xác tuyệt đối |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(list(accumulate([9, 1, 2, 3, 9], initial=0)))
print((15 - 9) / (4 - 1), (15 - 9) // (4 - 1))
print(Solution().largestSumOfAverages([4, 4, 4], 3))
```

In [35]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [1, 2, 3, 4, 5, 6, 7]` và `k = 4`: tính điểm của hai cách chia `[1,2,3,4] [5] [6] [7]` và `[1,2] [3,4] [5,6] [7]`, rồi cho biết cách nào bằng đáp án `20.5`.

In [36]:
# Bài 2: điền kết quả tính tay
hand_score_a = None     # điểm của [1,2,3,4] [5] [6] [7]
hand_score_b = None     # điểm của [1,2] [3,4] [5,6] [7]
hand_best = None        # "a" hoặc "b"

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DP bảng 2 chiều.

In [37]:
def exercise_3_table(nums: List[int], k: int) -> float:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    f = [[0.0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        # TODO 1: đặt f[i][1] bằng trung bình của i phần tử đầu
        for j in range(2, min(i, k) + 1):
            # TODO 2: duyệt h từ j - 1 tới i - 1
            # TODO 3: f[i][j] = max(f[i][j], f[h][j - 1] + trung bình của đoạn [h, i))
            pass
    # TODO 4: trả về f[n][k]
    return None


# assert math.isclose(exercise_3_table([9, 1, 2, 3, 9], 3), 20.0, abs_tol=1e-6)
# assert math.isclose(exercise_3_table([1, 2, 3, 4, 5, 6, 7], 4), 20.5, abs_tol=1e-6)
# assert math.isclose(exercise_3_table([5], 1), 5.0, abs_tol=1e-6)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_memo(nums, k)` bằng đệ quy có nhớ `dfs(i, k)` trên hậu tố.

In [38]:
def exercise_4_memo(nums: List[int], k: int) -> float:
    # TODO
    pass


# assert math.isclose(exercise_4_memo([9, 1, 2, 3, 9], 3), 20.0, abs_tol=1e-6)
# assert math.isclose(exercise_4_memo([1, 2, 3, 4, 5, 6, 7], 4), 20.5, abs_tol=1e-6)
# assert math.isclose(exercise_4_memo([1, 2, 3], 1), 2.0, abs_tol=1e-6)

### Bài 5 — Mini challenge: trả về cách chia tối ưu (khó)

Viết `exercise_5_partition(nums, k)` trả về `(điểm, danh_sách_các_đoạn)` với `danh_sách_các_đoạn` là list các list (đoạn liên tiếp, đủ mọi phần tử). Có thể có nhiều cách chia cùng điểm tối ưu, nên kiểm tra bằng **tính hợp lệ**: các đoạn ghép lại bằng `nums`, tối đa `k` đoạn, và điểm của cách chia (tính bằng `Fraction`) bằng điểm tối ưu.

```python
exercise_5_partition([9, 1, 2, 3, 9], 3)  -> (20.0, [[9], [1, 2, 3], [9]])
```
Gợi ý: lưu `choice[i][j]` = điểm cắt tốt nhất và đi ngược.

In [39]:
def exercise_5_partition(nums: List[int], k: int):
    # TODO
    pass


def valid_partition(nums, k, parts) -> bool:
    flat = [x for p in parts for x in p]
    return flat == nums and 1 <= len(parts) <= k and all(len(p) > 0 for p in parts)

def partition_score(parts) -> Fraction:
    return sum((Fraction(sum(p), len(p)) for p in parts), Fraction(0))


# for nums, k in [([9, 1, 2, 3, 9], 3), ([1, 2, 3, 4, 5, 6, 7], 4), ([5], 1)]:
#     value, parts = exercise_5_partition(nums, k)
#     assert valid_partition(nums, k, parts)
#     assert partition_score(parts) == oracle(nums, k)
#     assert math.isclose(value, float(oracle(nums, k)), abs_tol=1e-6)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `accumulate(..., initial=0)` thêm số 0 ở đầu.
- Mức 2: tổng đoạn `[1, 4)` là `s[4] - s[1]`; `/` cho số thực, `//` cho số nguyên.
- Mức 3: `[0, 9, 10, 12, 15, 24]`, `2.0 2`, `12.0` (ba đoạn `[4] [4] [4]`).

**Bài 2**
- Mức 1: trung bình `[1,2,3,4]` là `2,5`.
- Mức 2: cách a: `2,5 + 5 + 6 + 7`; cách b: `1,5 + 3,5 + 5,5 + 7`.
- Mức 3: a = `20,5`, b = `17,5`; đáp án đúng là cách a.

**Bài 3**
- Mức 1: `f[i][1] = s[i] / i`.
- Mức 2: `for h in range(j - 1, i)`.
- Mức 3: `f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))`; `return f[n][k]`.

**Bài 4**
- Mức 1: `@cache def dfs(i, k)`; `i == n` trả 0; `k == 1` trả trung bình hậu tố.
- Mức 2: vòng `for j in range(i + 1, n)`: `(s[j] - s[i]) / (j - i) + dfs(j, k - 1)`.
- Mức 3: `return dfs(0, k)`.

**Bài 5**
- Mức 1: dùng DP đúng `k` đoạn (giá trị `-inf` cho trạng thái không hợp lệ).
- Mức 2: khi cập nhật `f[i][j]` bằng `h` tốt hơn thì lưu `choice[i][j] = h`.
- Mức 3: đi ngược `i = choice[i][j]; j -= 1` rồi đảo danh sách các biên; cắt `nums` theo các biên.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
print(list(accumulate([9, 1, 2, 3, 9], initial=0)))
print((15 - 9) / (4 - 1), (15 - 9) // (4 - 1))
print(Solution().largestSumOfAverages([4, 4, 4], 3))

assert list(accumulate([9, 1, 2, 3, 9], initial=0)) == [0, 9, 10, 12, 15, 24]
assert (15 - 9) / (4 - 1) == 2.0 and (15 - 9) // (4 - 1) == 2
assert math.isclose(Solution().largestSumOfAverages([4, 4, 4], 3), 12.0, abs_tol=1e-9)

[0, 9, 10, 12, 15, 24]
2.0 2
12.0


### Đáp án Bài 2

Cách a: `[1,2,3,4]` (2,5) + `5` + `6` + `7` = **20,5**. Cách b: `1,5 + 3,5 + 5,5 + 7` = 17,5. Đáp án đúng là cách a.

In [41]:
a = Fraction(10, 4) + 5 + 6 + 7
b = Fraction(3, 2) + Fraction(7, 2) + Fraction(11, 2) + 7
hand_score_a_solution, hand_score_b_solution, hand_best_solution = float(a), float(b), "a"
assert (hand_score_a_solution, hand_score_b_solution) == (20.5, 17.5)
assert a == oracle([1, 2, 3, 4, 5, 6, 7], 4) and b < a
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_table_solution(nums: List[int], k: int) -> float:
    n = len(nums)
    s = list(accumulate(nums, initial=0))
    f = [[0.0] * (k + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        f[i][1] = s[i] / i
        for j in range(2, min(i, k) + 1):
            for h in range(j - 1, i):
                f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
    return f[n][k]

assert math.isclose(exercise_3_table_solution([9, 1, 2, 3, 9], 3), 20.0, abs_tol=1e-6)
assert math.isclose(exercise_3_table_solution([1, 2, 3, 4, 5, 6, 7], 4), 20.5, abs_tol=1e-6)
assert math.isclose(exercise_3_table_solution([5], 1), 5.0, abs_tol=1e-6)
rng = random.Random(3)
for _ in range(100):
    n = rng.randint(1, 9)
    nums = [rng.randint(1, 50) for _ in range(n)]
    k = rng.randint(1, n)
    assert close(exercise_3_table_solution(nums, k), oracle(nums, k), 1e-6), (nums, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_memo_solution(nums: List[int], k: int) -> float:
    n = len(nums)
    s = list(accumulate(nums, initial=0))

    @cache
    def dfs(i: int, k: int) -> float:
        if i == n:
            return 0
        if k == 1 or n - i == 1:                       # một đoạn, hoặc chỉ còn một phần tử
            return (s[n] - s[i]) / (n - i)
        return max((s[j] - s[i]) / (j - i) + dfs(j, k - 1) for j in range(i + 1, n))

    return dfs(0, k)

assert math.isclose(exercise_4_memo_solution([9, 1, 2, 3, 9], 3), 20.0, abs_tol=1e-6)
assert math.isclose(exercise_4_memo_solution([1, 2, 3, 4, 5, 6, 7], 4), 20.5, abs_tol=1e-6)
assert math.isclose(exercise_4_memo_solution([1, 2, 3], 1), 2.0, abs_tol=1e-6)
rng = random.Random(4)
for _ in range(100):
    n = rng.randint(1, 9)
    nums = [rng.randint(1, 50) for _ in range(n)]
    k = rng.randint(1, n)
    assert close(exercise_4_memo_solution(nums, k), oracle(nums, k), 1e-6), (nums, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_partition_solution(nums: List[int], k: int):
    value, bounds = best_partition(nums, k)
    parts = [nums[bounds[i]:bounds[i + 1]] for i in range(len(bounds) - 1)]
    return value, parts

for nums, k in [([9, 1, 2, 3, 9], 3), ([1, 2, 3, 4, 5, 6, 7], 4), ([5], 1)]:
    value, parts = exercise_5_partition_solution(nums, k)
    assert valid_partition(nums, k, parts)
    assert partition_score(parts) == oracle(nums, k)
    assert math.isclose(value, float(oracle(nums, k)), abs_tol=1e-6)
rng = random.Random(5)
for _ in range(150):
    n = rng.randint(1, 9)
    nums = [rng.randint(1, 20) for _ in range(n)]
    k = rng.randint(1, n)
    value, parts = exercise_5_partition_solution(nums, k)
    assert valid_partition(nums, k, parts), (nums, k, parts)
    assert partition_score(parts) == oracle(nums, k), (nums, k, parts)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DP bảng với tổng tiền tố)

```python
class Solution:
    def largestSumOfAverages(self, nums: List[int], k: int) -> float:
        from itertools import accumulate
        n = len(nums)
        s = list(accumulate(nums, initial=0))            # s[i] = tổng i phần tử đầu
        f = [[0.0] * (k + 1) for _ in range(n + 1)]      # f[i][j]: i phần tử đầu, j đoạn
        for i in range(1, n + 1):
            f[i][1] = s[i] / i                           # một đoạn duy nhất
            for j in range(2, min(i, k) + 1):
                for h in range(j - 1, i):                # đoạn cuối là [h, i)
                    f[i][j] = max(f[i][j], f[h][j - 1] + (s[i] - s[h]) / (i - h))
        return f[n][k]
```

Công thức nhớ nhanh: `f[i][j] = max_h ( f[h][j-1] + (s[i] - s[h]) / (i - h) )`; luôn dùng đúng `k` đoạn (chia thêm không làm điểm giảm); so sánh số thực bằng dung sai.

### Checklist trước khi submit

- [ ] Dùng `/` (chia thực), không phải `//`?
- [ ] Độ dài đoạn nửa mở `[h, i)` là `i - h`, tổng là `s[i] - s[h]`?
- [ ] Đã khởi tạo `f[i][1] = s[i] / i`?
- [ ] Chỉ số `h` chạy từ `j - 1` tới `i - 1` (đủ phần tử cho `j - 1` đoạn đầu)?
- [ ] Không dùng tham lam "chia chỗ lợi nhất"?
- [ ] Khi test, so sánh số thực bằng `math.isclose`/dung sai (hoặc `Fraction`), không `==`?
- [ ] Đã thử `k = 1`, `k = n`, mọi phần tử bằng nhau, `n = 100` giá trị `10^4`?
- [ ] Tên method đúng `largestSumOfAverages(self, nums, k)` và trả `float`?